In [70]:
import pandas as pd
import numpy as np
import ruptures as rpt
from tqdm import tqdm
from pathlib import Path
import json
from sklearn.preprocessing import RobustScaler

In [25]:
# --- CONFIG ---
INPUT_FILE = Path("../data/processed/all_emotions_consolidated.csv")
OUTPUT_FILE = Path("../data/processed/emotion_change_points.csv")
EMOTION_COLUMNS = [
    'admiration','amusement','anger','annoyance','approval','caring','confusion','curiosity','desire',
    'disappointment','disapproval','disgust','embarrassment','excitement','fear','gratitude','grief','joy',
    'love','nervousness','optimism','pride','realization','relief','remorse','sadness','surprise','neutral'
]

In [26]:
# --- Load Data ---
df = pd.read_csv(INPUT_FILE)
print(f"Loaded data with {len(df)} videos")

Loaded data with 25992 videos


In [27]:
# --- Preprocess List Columns ---
for emo in EMOTION_COLUMNS:
    df[emo] = df[emo].apply(lambda x: np.array(eval(x)))

In [71]:
# --- Change Point Detection Function ---
def detect_change_points_multivariate(emotion_matrix, model="rbf", pen=10):
    emotion_matrix = RobustScaler().fit_transform(emotion_matrix)
    algo = rpt.Pelt(model=model, jump=2).fit(emotion_matrix)
    cps = algo.predict(pen=pen)
    # Filter short segments
    filtered_cps = [cps[0]] if cps else []
    for i in range(1, len(cps)):
        if cps[i] - cps[i-1] > 5:
            filtered_cps.append(cps[i])
    return filtered_cps

In [72]:
# --- Detect Change Points ---
segment_counts = []
change_points_list = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Detecting change points"):
    video_id = row['video_id']
    emotion_matrix = np.stack([row[emo] for emo in EMOTION_COLUMNS], axis=1)  # shape: (100, 28)
    try:
        cps = detect_change_points_multivariate(emotion_matrix, model="rbf", pen=1)
        segment_counts.append(len(cps))
        change_points_list.append({"video_id": video_id, "change_points": cps})
    except Exception as e:
        print(f"Error for video_id {video_id}: {e}")
        change_points_list.append({"video_id": video_id, "change_points": []})

Detecting change points: 100%|██████████| 25992/25992 [01:35<00:00, 273.36it/s]


In [73]:
change_points_df = pd.DataFrame(change_points_list)

print(f"Average number of change points per video: {np.mean(segment_counts):.2f}")

Average number of change points per video: 7.93


In [74]:
change_points_df

,video_id,change_points
0,wL8qX260NTk,"[6, 34, 40, 50, 64, 76]"
1,I61WP51sSsc,"[4, 14, 36, 46, 58, 74, 86, 94]"
2,bMmAI0neLaY,"[8, 42, 56, 62, 92]"
3,t58X0md283Y,"[4, 20, 28, 36, 52, 66, 72, 98]"
4,o8NiE3XMPrM,"[16, 36, 50, 60, 72, 86, 100]"
...,...,...
25987,cc-ZTaw4Phk,"[6, 12, 18, 32, 52, 76, 82]"
25988,eRCSRe-8pKo,"[2, 24, 36, 52, 62, 72, 98]"
25989,iTDx12l0T3M,"[6, 12, 18, 42, 76, 82, 96]"
25990,YJ9B_UfTFMw,"[2, 16, 28, 52, 62, 72, 94]"


In [75]:
change_points_df.to_csv("../data/processed/emotion_change_points.csv", index=False)